# Seq2Seq, Machine Translation

Among the major breakthroughs that prompted widespread interest in modern RNNs was a major advance in the applied field of statistical machine translation. Here, the model is presented with a sentence in one language and must predict the corresponding sentence in another. Note that here the sentences may be of different lengths, and that corresponding words in the two sentences may not occur in the same order, owing to differences in the two language’s grammatical structure.

Many problems have this flavor of mapping between two such “unaligned” sequences. Examples include mapping from dialog prompts to replies or from questions to answers. Broadly, such problems are called sequence-to-sequence (seq2seq) problems and they are our focus for both the remainder of this chapter and much of Section 11.



In [12]:
import os
import zipfile
import requests
import hashlib
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from typing import List, Tuple, Optional, Dict

class MTFraEng:
    """English-French Machine Translation Dataset."""
    
    DATA_URL = "http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip"
    EXPECTED_SHA1 = "94646ad1522d915e7b0f9296181140edcf86a4f5"
    
    def __init__(self, root="../data", num_workers=4, batch_size=32, max_examples=None):
        self.root = root
        self.num_workers = num_workers
        self.batch_size = batch_size
        self.max_examples = max_examples
        
        # Create root directory if it doesn't exist
        os.makedirs(root, exist_ok=True)
        
        # Initialize vocabularies
        self.src_vocab = None
        self.tgt_vocab = None
        
    def _build_vocab(self, tokens_list: List[List[str]], min_freq: int = 2) -> Dict[str, int]:
        """Build vocabulary from list of token lists.
        
        Args:
            tokens_list: List of token lists
            min_freq: Minimum frequency for a token to be included
            
        Returns:
            Dictionary mapping tokens to indices
        """
        # Count token frequencies
        token_freqs = {}
        for tokens in tokens_list:
            for token in tokens:
                token_freqs[token] = token_freqs.get(token, 0) + 1
                
        # Create vocabulary with special tokens
        vocab = {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}
        vocab.update({
            token: idx + len(vocab)
            for idx, (token, freq) in enumerate(token_freqs.items())
            if freq >= min_freq
        })
        
        return vocab

    def _preprocess(self, text: str) -> str:
        """Preprocess text by handling spaces and punctuation."""
        text = text.replace('\u202f', ' ').replace('\xa0', ' ')
        no_space = lambda char, prev_char: char in ',.!?' and prev_char != ' '
        out = [' ' + char if i > 0 and no_space(char, text[i - 1]) else char
               for i, char in enumerate(text.lower())]
        return ''.join(out)

    def _tokenize(self, text: str, max_examples: Optional[int] = None) -> Tuple[List[List[str]], List[List[str]]]:
        """Tokenize the text into source and target sequences."""
        src, tgt = [], []
        for i, line in enumerate(text.split('\n')):
            if max_examples and i > max_examples:
                break
            parts = line.split('\t')
            if len(parts) == 2:
                src.append([t for t in f'<bos> {parts[0]} <eos>'.split(' ') if t])
                tgt.append([t for t in f'<bos> {parts[1]} <eos>'.split(' ') if t])
        return src, tgt
            
    def _check_sha1(self, file_path: str) -> bool:
        """Check if the file's SHA1 hash matches the expected value."""
        sha1 = hashlib.sha1()
        with open(file_path, 'rb') as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        return sha1.hexdigest() == self.EXPECTED_SHA1

    def _download(self) -> str:
        """Download and extract the English-French dataset."""
        zip_path = os.path.join(self.root, "fra-eng.zip")
        
        if not os.path.exists(zip_path) or not self._check_sha1(zip_path):
            print(f"Downloading English-French dataset to {zip_path}...")
            response = requests.get(self.DATA_URL, stream=True)
            response.raise_for_status()
            
            with open(zip_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
                    
            if not self._check_sha1(zip_path):
                raise RuntimeError("Downloaded file has incorrect SHA1 hash")
                
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(self.root)
            
        txt_path = os.path.join(self.root, 'fra-eng', 'fra.txt')
        with open(txt_path, encoding='utf-8') as f:
            return f.read()

    def get_dataloader(self, train=True):
        """Get a DataLoader for the dataset."""
        # Download and preprocess the raw data
        raw_text = self._download()
        preprocessed_text = self._preprocess(raw_text)
        src_tokens, tgt_tokens = self._tokenize(preprocessed_text, self.max_examples)
        
        # Build vocabularies if not already built
        if self.src_vocab is None:
            self.src_vocab = self._build_vocab(src_tokens)
        if self.tgt_vocab is None:
            self.tgt_vocab = self._build_vocab(tgt_tokens)
        
        # Create dataset
        dataset = FraEngDataset(
            src_tokens=src_tokens,
            tgt_tokens=tgt_tokens,
            src_vocab=self.src_vocab,
            tgt_vocab=self.tgt_vocab,
            train=train
        )
        
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=train,
            num_workers=self.num_workers,
            collate_fn=self.collate_fn
        )
        
    @staticmethod
    def collate_fn(batch):
        """Custom collate function to handle variable length sequences."""
        # Separate source and target sequences
        src_seqs, tgt_seqs = zip(*batch)
        
        # Pad sequences
        src_padded = pad_sequence(src_seqs, batch_first=True, padding_value=0)
        tgt_padded = pad_sequence(tgt_seqs, batch_first=True, padding_value=0)
        
        return src_padded, tgt_padded
        
    def train_dataloader(self):
        """Get the training DataLoader."""
        return self.get_dataloader(train=True)
        
    def val_dataloader(self):
        """Get the validation DataLoader."""
        return self.get_dataloader(train=False)

    def decode_tokens(self, token_ids: torch.Tensor, is_source: bool = True) -> str:
        """Convert a sequence of token IDs back to text.
        
        Args:
            token_ids: Tensor of token IDs
            is_source: If True, use source vocabulary, else target vocabulary
            
        Returns:
            Decoded text string
        """
        # Choose the appropriate vocabulary
        vocab = self.src_vocab if is_source else self.tgt_vocab
        # Create reverse vocabulary (id to token mapping)
        id_to_token = {idx: token for token, idx in vocab.items()}
        
        # Convert to list if it's a tensor
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()
            
        # Convert IDs to tokens, ignoring padding and handling special tokens
        tokens = []
        for tid in token_ids:
            token = id_to_token.get(tid, '<unk>')
            # Skip padding and special tokens if desired
            if token not in ['<pad>', '<bos>', '<eos>']:
                tokens.append(token)
                
        # Join tokens with spaces
        return ' '.join(tokens)

    def decode_batch(self, batch_ids: torch.Tensor, is_source: bool = True) -> List[str]:
        """Convert a batch of token IDs back to texts.
        
        Args:
            batch_ids: Tensor of shape [batch_size, sequence_length]
            is_source: If True, use source vocabulary, else target vocabulary
            
        Returns:
            List of decoded strings
        """
        return [self.decode_tokens(seq, is_source) for seq in batch_ids]



class FraEngDataset(Dataset):
    """Custom Dataset for the English-French translation data."""
    
    def __init__(self, src_tokens: List[List[str]], tgt_tokens: List[List[str]], 
                 src_vocab: Dict[str, int], tgt_vocab: Dict[str, int],
                 train: bool = True):
        """Initialize the dataset."""
        assert len(src_tokens) == len(tgt_tokens)
        
        # Split into train/val (90/10 split)
        n = len(src_tokens)
        split = int(n * 0.9)
        
        if train:
            self.src_tokens = src_tokens[:split]
            self.tgt_tokens = tgt_tokens[:split]
        else:
            self.src_tokens = src_tokens[split:]
            self.tgt_tokens = tgt_tokens[split:]
            
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        
    def __len__(self):
        return len(self.src_tokens)
    
    def __getitem__(self, idx):
        """Return a single item from the dataset."""
        # Convert tokens to indices
        src = torch.tensor([self.src_vocab.get(token, self.src_vocab['<unk>']) 
                          for token in self.src_tokens[idx]], dtype=torch.long)
        tgt = torch.tensor([self.tgt_vocab.get(token, self.tgt_vocab['<unk>']) 
                          for token in self.tgt_tokens[idx]], dtype=torch.long)
        
        return src, tgt

In [13]:
# Initialize dataset with maximum number of examples
mt_dataset = MTFraEng(
    root="./data",
    num_workers=0,
    batch_size=32,
    max_examples=1000  # Limit to first 1000 examples
)

# Get train and validation dataloaders
train_loader = mt_dataset.train_dataloader()
val_loader = mt_dataset.val_dataloader()

# Example iteration
for src_batch, tgt_batch in train_loader:
    # src_batch and tgt_batch will be lists of tokenized sentences
    print(f"Source batch size: {len(src_batch)}")
    print(f"Target batch size: {len(tgt_batch)}")
    break

Source batch size: 32
Target batch size: 32


In [14]:
src_batch

tensor([[  4, 197,  52,   6,   7,   0],
        [  4, 251,  33, 333,  12,   7],
        [  4,  43, 330,   6,   7,   0],
        [  4,   1,  10,   7,   0,   0],
        [  4,  52,  40,   6,   7,   0],
        [  4, 204,  33,   6,   7,   0],
        [  4,  21, 260, 102,   6,   7],
        [  4,  29, 116, 130,   6,   7],
        [  4,  43, 305,   6,   7,   0],
        [  4,  26,  47,  10,   7,   0],
        [  4,  54, 118,   6,   7,   0],
        [  4, 194, 114,  10,   7,   0],
        [  4,  43, 116,   1,   6,   7],
        [  4, 216, 217,  12,   7,   0],
        [  4,  21, 289,  30,   6,   7],
        [  4,  68, 126,   6,   7,   0],
        [  4, 114, 115,  10,   7,   0],
        [  4,  21, 112,  87,   6,   7],
        [  4,  43, 161,   6,   7,   0],
        [  4,  23, 212,   6,   7,   0],
        [  4,  21,  38,   6,   7,   0],
        [  4, 216, 217,  12,   7,   0],
        [  4,  11,  12,   7,   0,   0],
        [  4,  21, 112, 161,   6,   7],
        [  4,  21, 144,   6,   7,   0],


In [15]:
tgt_batch

tensor([[  4,   1, 106,  20,   7,   0,   0,   0],
        [  4, 494,   1,  12,   7,   0,   0,   0],
        [  4,  30,  63, 672,  17, 674,  20,   7],
        [  4,  17, 152,   1,  20,   7,   0,   0],
        [  4, 106,   1,  20,   7,   0,   0,   0],
        [  4,   1,  20,   7,   0,   0,   0,   0],
        [  4,  30, 157,  80, 536,  20,   7,   0],
        [  4, 268, 160, 286,   6,   7,   0,   0],
        [  4,  30,  63,   1,  20,   7,   0,   0],
        [  4,  88,   1,  90,  77,  87,   6,   7],
        [  4, 108, 109,   6,   7,   0,   0,   0],
        [  4, 400, 729, 156,   6,   7,   0,   0],
        [  4,  30,  63,   1,  20,   7,   0,   0],
        [  4, 463, 464, 142,  12,   7,   0,   0],
        [  4,  30, 182,  63, 596,  20,   7,   0],
        [  4,   1,   6,   7,   0,   0,   0,   0],
        [  4, 252,  20,   7,   0,   0,   0,   0],
        [  4,  30,  63, 200,  20,   7,   0,   0],
        [  4,  30,  63, 341,  20,   7,   0,   0],
        [  4,   1,   6,   7,   0,   0,   0,   0],


In [7]:
import pandas as pd

pd.DataFrame({"source": mt_dataset.decode_batch(src_batch), "target": mt_dataset.decode_batch(tgt_batch, is_source=False)})

,source,target
0,i'm a <unk> .,je suis <unk> .
1,let me go .,laissez-moi m'en aller !
2,i'm <unk> .,j'ai <unk> .
3,it's hers .,c'est la sienne .
4,shut up !,ferme-la !
5,she cried .,elle pleura .
6,i talked .,j’ai parlé .
7,come on .,venez !
8,taste it .,<unk> .
9,kiss tom .,<unk> tom .
